# XGBoost Model for Delinquency Prediction

Uses the **top N selected features** from the comprehensive feature selection pipeline.

- Train / Validation / Test split (60/20/20)
- XGBoost binary classifier with early stopping
- Handles class imbalance with `scale_pos_weight`
- No feature scaling required for tree-based models
- Records training time, scoring time, ROC-AUC, and Average Precision

In [1]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, average_precision_score, roc_curve,
)
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from scripts.model_data import load_and_split, save_sorted_features

## 1. Load Data & Selected Features

In [2]:
# ── Configuration ────────────────────────────────────────────────
N_FEATURES = 50  # ← change this to use a different number of top-ranked features

# Write output/sorted_features.csv (columns ordered by rank) once
save_sorted_features()

# Load features and perform 60/20/20 stratified split
X_train, X_val, X_test, y_train, y_val, y_test, available_features, features_df = \
    load_and_split(n_features=N_FEATURES)

print(f'Features used:  {len(available_features)}')
print(f'Train:          {X_train.shape}  |  positives: {y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Validation:     {X_val.shape}  |  positives: {y_val.sum()} ({100*y_val.mean():.1f}%)')
print(f'Test:           {X_test.shape}  |  positives: {y_test.sum()} ({100*y_test.mean():.1f}%)')

Saved sorted_features.csv  (173 ranked + 45 other columns)
Features  : 50 (top-50)
Samples   : 10317  |  DQ rate: 8.86%
Train     : 6190  (8.85% positive)
Val       : 2063  (8.87% positive)
Test      : 2064  (8.87% positive)
Features used:  50
Train:          (6190, 50)  |  positives: 548 (8.9%)
Validation:     (2063, 50)  |  positives: 183 (8.9%)
Test:           (2064, 50)  |  positives: 183 (8.9%)


## 2. Prepare DMatrix Objects

XGBoost uses its own `DMatrix` format. No scaling is needed for tree-based models.

In [3]:
# Compute scale_pos_weight to handle class imbalance
n_neg = int(np.sum(y_train == 0))
n_pos = int(np.sum(y_train == 1))
scale_pos_weight = n_neg / n_pos
print(f'scale_pos_weight: {scale_pos_weight:.2f}  (neg={n_neg}, pos={n_pos})')

# Build DMatrix objects (XGBoost native format)
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(available_features))
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=list(available_features))
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=list(available_features))

print('\nDMatrix objects created successfully.')

scale_pos_weight: 10.30  (neg=5642, pos=548)

DMatrix objects created successfully.


## 3. Hyperparameter Tuning with Optuna

Use **Optuna** (TPE sampler) to minimise overfitting and maximise validation AUC.  
The search targets the key regularisation knobs: tree depth, leaf weight, gamma, L1/L2, and sampling rates.

In [4]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS      = 80    # increase for a more thorough search
EARLY_STOPPING = 50
NUM_BOOST_ROUND = 2000

def objective(trial):
    p = {
        'objective':         'binary:logistic',
        'eval_metric':       'auc',
        'scale_pos_weight':  scale_pos_weight,
        'seed':              42,
        'verbosity':         0,
        # --- tree complexity ---
        'max_depth':         trial.suggest_int('max_depth', 2, 6),
        'min_child_weight':  trial.suggest_int('min_child_weight', 5, 50),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        # --- regularisation ---
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1.0, 20.0),
        # --- stochastic sampling ---
        'eta':               trial.suggest_float('eta', 0.01, 0.1, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 0.9),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 0.9),
    }

    pruning_cb = optuna.integration.XGBoostPruningCallback(trial, 'val-auc')

    bst = xgb.train(
        p,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=EARLY_STOPPING,
        verbose_eval=False,
        callbacks=[pruning_cb],
    )
    return bst.best_score   # maximise val AUC

sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_warmup_steps=10)
study   = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)

print(f'Running Optuna search ({N_TRIALS} trials) …')
t_tune = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'Search finished in {time.time()-t_tune:.1f}s')
print(f'\nBest val AUC : {study.best_value:.4f}')
print(f'Best params  :')
for k, v in study.best_params.items():
    print(f'  {k:<22} {v}')

Running Optuna search (80 trials) …


  0%|          | 0/80 [00:00<?, ?it/s]

Search finished in 16.1s

Best val AUC : 0.8067
Best params  :
  max_depth              5
  min_child_weight       20
  gamma                  0.3177917514301182
  reg_alpha              3.109823217156622
  reg_lambda             7.178483118508193
  eta                    0.053654503243520245
  subsample              0.7550229885420853
  colsample_bytree       0.8436063712881633
  colsample_bylevel      0.6361074625809746


## 4. Train Final Model with Best Hyperparameters

In [5]:
params = {
    'objective':        'binary:logistic',
    'eval_metric':      ['auc', 'aucpr'],
    'scale_pos_weight': scale_pos_weight,
    'seed':             42,
    'verbosity':        1,
    **study.best_params,
}

evals = [(dtrain, 'train'), (dval, 'val')]
evals_result = {}

print(f'Training final XGBoost with best params (top-{N_FEATURES} features)…')
t0 = time.time()
booster = xgb.train(
    params,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=evals,
    early_stopping_rounds=EARLY_STOPPING,
    evals_result=evals_result,
    verbose_eval=100,
)
training_time = time.time() - t0

print(f'\nTraining completed in {training_time:.2f}s')
print(f'Best iteration : {booster.best_iteration}')
print(f'Best val AUC   : {booster.best_score:.4f}')

Training final XGBoost with best params (top-50 features)…
[0]	train-auc:0.80205	train-aucpr:0.27255	val-auc:0.72441	val-aucpr:0.19807
[100]	train-auc:0.94154	train-aucpr:0.59886	val-auc:0.79054	val-aucpr:0.26347
[200]	train-auc:0.97685	train-aucpr:0.78363	val-auc:0.79789	val-aucpr:0.29382
[300]	train-auc:0.99027	train-aucpr:0.89553	val-auc:0.80201	val-aucpr:0.30797
[400]	train-auc:0.99477	train-aucpr:0.94256	val-auc:0.80492	val-aucpr:0.31378
[476]	train-auc:0.99668	train-aucpr:0.96288	val-auc:0.80538	val-aucpr:0.31118

Training completed in 0.99s
Best iteration : 426
Best val AUC   : 0.3173


## 9. Summary Table (Top-N Features)

In [10]:
print(f'XGBoost – Top-{N_FEATURES} Features Summary')
print('=' * 60)
print(f'Objective:       binary:logistic')
print(f'max_depth:       {params["max_depth"]}  |  eta: {params["eta"]}')
print(f'subsample:       {params["subsample"]}  |  colsample_bytree: {params["colsample_bytree"]}')
print(f'Best iteration:  {booster.best_iteration}')
print(f'Features used:   {len(available_features)}')
print(f'Training Time:   {training_time:.2f}s ({training_time/60:.2f} min)')
print()
print(f'{"Split":<12} {"ROC-AUC":<12} {"Avg Precision":<16} {"Scoring Time":<14}')
print(f'{"-"*54}')
print(f'{"Train":<12} {train_auc:<12.4f} {train_ap:<16.4f} {train_time:<14.4f}s')
print(f'{"Val":<12} {val_auc:<12.4f} {val_ap:<16.4f} {val_time:<14.4f}s')
print(f'{"Test":<12} {test_auc:<12.4f} {test_ap:<16.4f} {test_time:<14.4f}s')

XGBoost – Top-50 Features Summary
Objective:       binary:logistic
max_depth:       5  |  eta: 0.053654503243520245
subsample:       0.7550229885420853  |  colsample_bytree: 0.8436063712881633
Best iteration:  426
Features used:   50
Training Time:   0.99s (0.02 min)

Split        ROC-AUC      Avg Precision    Scoring Time  
------------------------------------------------------
Train        0.9967       0.9628           0.0060        s
Val          0.8054       0.3141           0.0010        s
Test         0.7770       0.2694           0.0008        s
